# Monarch Admin TUI: Complete Guide

This notebook covers everything about the **Mesh Admin TUI** (`monarch-tui`) —
an interactive terminal client for inspecting live Monarch meshes in real time.

---

## 1. What Is the Admin TUI?

The **Mesh Admin TUI** is a full-featured, interactive terminal application that lets you
**inspect, navigate, and diagnose live Monarch meshes** in real time.

It connects to the mesh admin HTTP API and renders the full topology — hosts, processes,
and actors — as a navigable tree with a contextual detail pane.

| Aspect | Details |
|--------|--------|
| **Binary name** | `monarch-tui` |
| **Distribution** | Included in `torchmonarch` PyPI package (`pip install torchmonarch`) |
| **Implementation** | Rust crate `hyperactor_mesh_admin_tui` |
| **Terminal stack** | `ratatui` v0.30 + `crossterm` v0.28 |
| **Protocol** | HTTP GET to `MeshAdminAgent` endpoints |
| **OSS status** | Yes — fully open source, works with OSS Monarch |

---
## 2. Does It Work with OSS?

**Yes.** The Admin TUI is fully open source and works with OSS Monarch installations.

- The binary entry point (`hyperactor_mesh_admin_tui/bin/admin_tui.rs`) has a `#[cfg(not(fbcode_build))]`
  path that uses standard `#[tokio::main]`, no internal dependencies required.
- It is distributed via `pip install torchmonarch`, making `monarch-tui` available on PATH.
- TLS is optional and auto-detected — plain HTTP works out of the box for local development.
- The entire crate uses only open-source dependencies: `ratatui`, `crossterm`, `reqwest`, `clap`,
  `tokio`, `serde_json`, `chrono`, etc.

```bash
# Install
pip install torchmonarch

# Run
monarch-tui --addr 127.0.0.1:1729
```

---
## 3. Architecture: How It Works

### 3.1 The Big Picture

The TUI is a **lazy reference walker**. It does not have privileged topology knowledge.
It starts at `root`, follows `children` references, and asks the server to resolve
whatever opaque reference string it was given next.

```
TUI (terminal client)
  |
  |  GET /v1/{reference}
  v
MeshAdminAgent (HTTP bridge + resolver)
  |
  |-- resolve_reference("root")     -> build_root_payload()
  |-- resolve_reference("host:...") -> HostAgent introspect query
  |-- resolve_reference(proc_id)    -> HostAgent or ProcAgent query
  |-- resolve_reference(actor_id)   -> direct actor introspect query
  v
NodePayload (uniform response)
  -> TUI cache + tree builder
  -> visible rows in the tree
```

### 3.2 The Uniform Data Model: `NodePayload`

The entire admin surface is built around **one response type**:

```rust
pub struct NodePayload {
    pub identity: String,            // canonical reference for this node
    pub properties: NodeProperties,  // Root | Host | Proc | Actor | Error
    pub children: Vec<String>,       // next references to walk
    pub parent: Option<String>,      // upward navigation hint
    pub as_of: String,               // timestamp of data capture
}
```

Clients never need separate "host response" or "actor response" shapes. Everything
is "resolve this reference" and receive a `NodePayload`.

### 3.3 The Topology Tree

```
root (synthetic navigation anchor)
  +-- host:<host_agent_id_0>
  |     +-- service_proc
  |     |     +-- MeshAdminAgent
  |     |     +-- HostAgent
  |     +-- user_proc_0
  |     |     +-- ProcAgent
  |     |     +-- MyTrainer[0]
  |     |     +-- MyTrainer[1]
  |     +-- user_proc_1
  |           +-- ProcAgent
  |           +-- MyGenerator[0]
  +-- host:<host_agent_id_1>
        +-- ...
```

### 3.4 Three-Layer Resolution

| Layer | Component | Provides |
|-------|-----------|----------|
| 1 | **MeshAdminAgent** | Mesh-wide resolver, HTTP bridge, synthetic root node |
| 2 | **HostAgent** | Host nodes, system/local proc synthesis |
| 3 | **ProcAgent** | User proc nodes, terminated actor snapshots, py-spy, config dump |

### 3.5 Event Loop Architecture

The TUI main loop (`run_app()` in `app.rs`) uses `tokio::select!` to multiplex three event sources:

| Source | Purpose |
|--------|---------|
| **Refresh timer** | Periodic topology rebuild (default every 2s) |
| **Terminal events** | Keyboard input via crossterm `EventStream` |
| **Active job results** | Diagnostics stream, py-spy results, config dump results |

### 3.6 Algebraic Design Invariants

The TUI is structured around **21+ formally stated invariants** for correctness:

| Invariant | Rule |
|-----------|------|
| **TUI-1** | All fetch results merge via join-semilattice (commutative, associative, idempotent) |
| **TUI-2** | Cursor enforces `pos < len` (or `pos == 0` when empty) at all times |
| **TUI-3** | Topology stored as explicit `TreeNode { children }`, rendered via pure `flatten_tree()` |
| **TUI-4** | Failed nodes always visible regardless of `show_stopped` toggle |
| **TUI-5** | All cache writes go through `fetch_with_join()` — no direct inserts |
| **TUI-8** | Tree building rejects true cycles (nodes in their own ancestor path) |
| **TUI-10** | All tree walks use fold abstractions, never ad-hoc recursion |
| **TUI-21** | `active_job.is_some()` if and only if `overlay.is_some()` |

---
## 4. Core Features

### 4.1 Interactive Topology Tree

The main view is a **split-pane layout**:
- **Left pane (40%):** Expandable topology tree with hosts, procs, actors
- **Right pane (60%):** Contextual detail for the selected node
- **Header (3 rows):** Connection info, uptime, filter state, theme/lang
- **Footer (2 rows):** Context-sensitive keybinding hints

Navigate with `j`/`k` (vim-style), expand/collapse with `Tab`.

When you select an **actor**, the detail pane shows:
- Actor status (running / stopped / failed)
- Actor type
- Message count
- Processing time
- Last handler invoked
- **Flight recorder events** (recent activity log)

### 4.2 Diagnostics Overlay

Press `d` to run a **full health check** across the entire mesh.

The diagnostics engine walks the full resolution graph:
```
root -> hosts -> service procs -> system actors -> user procs -> user actors
```

Each node is probed via HTTP and reported as:
- **Pass** (healthy, fast response)
- **Slow** (reachable but latency above threshold)
- **Fail** (unreachable or error)

Results are separated into:
- **Admin Infrastructure** — admin server, host agents, service procs
- **Mesh** — user procs and actors

Each probe shows its latency in milliseconds.

### 4.3 Py-spy Stack Traces

Press `p` on any proc or actor to capture a **live Python stack trace** via
[py-spy](https://github.com/benfred/py-spy).

The overlay shows:
- Per-thread stack traces with frame-level detail
- GIL ownership flags
- Thread names
- Active/idle status

Each press fetches a **fresh trace** — useful for diagnosing hangs in C extensions and CUDA calls.

**Note:** `py-spy` is included as a default dependency of `torchmonarch`.

### 4.4 Config Dump

Press `C` (shift-c) to view the **process configuration snapshot** with source provenance
highlighting (shows where each config value came from).

### 4.5 Filtering

| Key | Filter | Description |
|-----|--------|-------------|
| `s` | System actors | Toggle visibility of infrastructure actors (MeshAdminAgent, HostAgent, etc.) |
| `h` | Stopped actors | Toggle visibility of stopped actors (**failed actors always remain visible** per TUI-4) |

### 4.6 Theming & Localization

| Option | Values |
|--------|--------|
| `--theme` | `nord` (dark, default) or `doom-nord-light` (light) |
| `--lang` | `en` (English, default) or `zh` (Simplified Chinese) |

### 4.7 Non-Interactive Diagnostics Mode

For **scripted health checks** and CI/CD integration:

```bash
monarch-tui --addr 127.0.0.1:1729 --diagnose
# Prints JSON report to stdout
# Exits 0 if healthy, 1 if any check failed
```

### 4.8 TLS/mTLS Support

TLS is **auto-detected** by default. Explicit configuration:

```bash
monarch-tui --addr host:port \
    --tls-ca /path/to/ca.pem \
    --tls-cert /path/to/client.pem \
    --tls-key /path/to/client.key
```

---
## 5. Complete Keybinding Reference

### Main View

| Key | Action |
|-----|--------|
| `j` / `Down` | Move cursor down |
| `k` / `Up` | Move cursor up |
| `g` / `Home` | Jump to top |
| `G` / `End` | Jump to bottom |
| `PgDn` / `Ctrl+D` / `Ctrl+V` | Page down (10 rows) |
| `PgUp` / `Ctrl+U` / `Alt+V` | Page up (10 rows) |
| `Tab` | Expand/collapse selected node |
| `c` | Collapse all nodes |
| `s` | Toggle system actor visibility |
| `h` | Toggle stopped actor visibility |
| `d` | Run diagnostics overlay |
| `p` | Py-spy stack trace for selected proc/actor |
| `C` (shift) | Config dump for selected proc |
| `Ctrl+L` | Scroll selected item to top of viewport |
| `Esc` | Dismiss overlay |
| `q` / `Ctrl+C` | Quit |

### Inside Overlays

| Key | Action |
|-----|--------|
| `j` / `k` | Scroll overlay content |
| `r` or `d` | Re-run diagnostics (in diagnostics overlay) |
| `p` | Re-run py-spy (in py-spy overlay) |
| `C` | Re-run config dump (in config overlay) |
| `Esc` | Dismiss overlay |

---
## 6. CLI Options Reference

```
monarch-tui [OPTIONS] --addr <ADDR>
```

| Flag | Description | Default |
|------|-------------|--------|
| `--addr`, `-a` | Admin server address (`host:port` or `https://host:port`) | **required** |
| `--refresh-ms` | Auto-refresh interval in milliseconds | `2000` |
| `--theme` | Color theme: `nord` (dark) or `doom-nord-light` (light) | `nord` |
| `--lang` | Display language: `en` or `zh` (Simplified Chinese) | `en` |
| `--diagnose` | Run diagnostics non-interactively, print JSON, exit | `false` |
| `--tls-ca` | Path to PEM CA certificate for TLS | auto-detected |
| `--tls-cert` | Path to PEM client certificate for mutual TLS | auto-detected |
| `--tls-key` | Path to PEM client private key for mutual TLS | — |

---
## 7. Quick Start: Step-by-Step Example

The easiest way to try the Admin TUI is with the **Dining Philosophers** example —
five philosopher actors share chopsticks around a table, mediated by a waiter actor.

### Step 1: Start the Monarch Application

In **Terminal 1**, run the dining philosophers example:

```bash
python python/examples/dining_philosophers.py
```

The example prints the admin server address on startup:

```
Mesh admin server listening on http://127.0.0.1:1729
  - Root node:     curl http://127.0.0.1:1729/v1/root
  - Mesh tree:     curl http://127.0.0.1:1729/v1/tree
  - API docs:      curl http://127.0.0.1:1729/SKILL.md
  - TUI:           monarch-tui --addr http://127.0.0.1:1729
```

### Step 2: Attach the TUI

In **Terminal 2**, connect the TUI:

```bash
monarch-tui --addr 127.0.0.1:1729
```

You'll see the topology tree:

```
+- host:...
   +- service_proc
   |  +- MeshAdminAgent
   |  +- HostAgent
   +- proc_0
   |  +- philosopher[0]
   |  +- philosopher[1]
   |  +- philosopher[2]
   |  +- philosopher[3]
   |  +- philosopher[4]
   +- waiter_proc
      +- waiter
```

### Step 3: Explore

- Use `j`/`k` to navigate, `Tab` to expand/collapse
- Select a philosopher actor to see its message count, processing time, flight recorder
- Press `d` for diagnostics — see if all actors are healthy
- Press `p` on a proc for a py-spy stack trace
- Press `s` to toggle system actors (MeshAdminAgent, HostAgent, ProcAgent)

### Step 4: Enabling the Admin Server in Your Own Code

To enable the admin TUI for your own Monarch application, add `MeshAdminConfig()`
to your `ProcessJob`:

In [ ]:
# How to enable the mesh admin server in your own Monarch application

from monarch.job import MeshAdminConfig, ProcessJob

# Add mesh_admin=MeshAdminConfig() to enable the admin HTTP API
job = ProcessJob(
    {"hosts": 1},
    mesh_admin=MeshAdminConfig(),  # <-- This is all you need!
)

state = job.state(cached_path=None)

# The admin URL is available via:
admin_url = state.admin_url
print(f"Admin TUI: monarch-tui --addr {admin_url}")

---
## 8. Full Dining Philosophers Example (with Admin TUI)

Below is the complete Python example from `python/examples/dining_philosophers.py`
that ships with Monarch. It demonstrates actors, coordination, and admin TUI integration.

In [ ]:
# Complete example from python/examples/dining_philosophers.py
#
# Five philosopher actors sit around a table.
# Each needs two chopsticks (shared with neighbors) to eat.
# A Waiter actor arbitrates access to prevent deadlock.
# MeshAdminAgent is spawned for TUI access.
#
# Run: python python/examples/dining_philosophers.py
# Then: monarch-tui --addr <printed_address>

import asyncio
from enum import auto, Enum
from typing import Any

from monarch.actor import Actor, current_rank, endpoint
from monarch.job import MeshAdminConfig, ProcessJob


class ChopstickStatus(Enum):
    NONE = auto()
    REQUESTED = auto()
    GRANTED = auto()


class Philosopher(Actor):
    """A philosopher that alternates between thinking and eating."""

    def __init__(self, size: int) -> None:
        self.size = size
        self.rank = 0
        self.left_status = ChopstickStatus.NONE
        self.right_status = ChopstickStatus.NONE
        self.waiter = None
        self.meals_eaten = 0

    def _chopstick_indices(self):
        left = self.rank % self.size
        right = (self.rank + 1) % self.size
        return left, right

    async def _request_chopsticks(self):
        left, right = self._chopstick_indices()
        self.left_status = ChopstickStatus.REQUESTED
        self.right_status = ChopstickStatus.REQUESTED
        await self.waiter.request_chopsticks.call_one(self.rank, left, right)

    async def _release_chopsticks(self):
        left, right = self._chopstick_indices()
        self.left_status = ChopstickStatus.NONE
        self.right_status = ChopstickStatus.NONE
        await self.waiter.release_chopsticks.call_one(left, right)

    @endpoint
    async def start(self, waiter: Any) -> None:
        self.rank = current_rank().rank
        self.waiter = waiter
        await self._request_chopsticks()

    @endpoint
    async def grant_chopstick(self, chopstick: int) -> None:
        left, right = self._chopstick_indices()
        if chopstick == left:
            self.left_status = ChopstickStatus.GRANTED
        elif chopstick == right:
            self.right_status = ChopstickStatus.GRANTED

        if (
            self.left_status == ChopstickStatus.GRANTED
            and self.right_status == ChopstickStatus.GRANTED
        ):
            self.meals_eaten += 1
            print(f"philosopher {self.rank} is eating (meal {self.meals_eaten})")
            await asyncio.sleep(1)
            await self._release_chopsticks()
            await asyncio.sleep(0.5)
            await self._request_chopsticks()


class Waiter(Actor):
    """Arbitrates chopstick access to prevent deadlock."""

    def __init__(self, philosophers: Any) -> None:
        self.philosophers = philosophers
        self.assignments = {}  # chopstick -> philosopher rank
        self.requests = {}    # chopstick -> waiting philosopher rank

    def _try_grant(self, rank, chopstick):
        if chopstick not in self.assignments:
            self.assignments[chopstick] = rank
            self.philosophers.slice(replica=rank).grant_chopstick.broadcast(chopstick)
        else:
            self.requests[chopstick] = rank

    def _release(self, chopstick):
        self.assignments.pop(chopstick, None)
        if chopstick in self.requests:
            rank = self.requests.pop(chopstick)
            self._try_grant(rank, chopstick)

    @endpoint
    async def request_chopsticks(self, rank, left, right):
        self._try_grant(rank, left)
        self._try_grant(rank, right)

    @endpoint
    async def release_chopsticks(self, left, right):
        self._release(left)
        self._release(right)


# --- Entry point ---
# async def main():
#     job = ProcessJob({"hosts": 1}, mesh_admin=MeshAdminConfig())
#     state = job.state(cached_path=None)
#     host = state.hosts
#
#     print(f"Admin TUI: monarch-tui --addr {state.admin_url}")
#
#     procs = host.spawn_procs(per_host={"replica": 5})
#     waiter_proc = host.spawn_procs(name="waiter")
#     philosophers = procs.spawn("philosopher", Philosopher, 5)
#     waiter = waiter_proc.spawn("waiter", Waiter, philosophers)
#     philosophers.start.broadcast(waiter)
#
#     await asyncio.sleep(float("inf"))  # Run until interrupted
#
# asyncio.run(main())

---
## 9. HTTP API Endpoints

The TUI communicates with the `MeshAdminAgent` via these HTTP endpoints:

| Endpoint | Purpose | Used by TUI |
|----------|---------|-------------|
| `GET /v1/root` | Synthetic root node with host children | Yes (start) |
| `GET /v1/{reference}` | Resolve any reference to `NodePayload` | Yes (navigation) |
| `GET /v1/tree` | Full pre-expanded tree | No (TUI walks lazily) |
| `GET /v1/pyspy/{proc_ref}` | Py-spy stack trace for a proc | Yes (`p` key) |
| `GET /v1/config/{proc_ref}` | Config dump for a proc | Yes (`C` key) |
| `GET /v1/schema` | JSON schema for `NodePayload` | No |
| `GET /v1/openapi.json` | OpenAPI specification | No |
| `GET /SKILL.md` | Machine-readable API contract | No |

You can also use these endpoints directly with `curl`:

```bash
# Get the root node
curl http://127.0.0.1:1729/v1/root

# Get the full mesh tree
curl http://127.0.0.1:1729/v1/tree

# Read the API docs
curl http://127.0.0.1:1729/SKILL.md
```

---
## 10. Benefits & Advantages

### Why is the Admin TUI Valuable?

| # | Benefit | Description |
|---|---------|-------------|
| 1 | **Real-time visibility** | See the live state of every host, proc, and actor in your mesh |
| 2 | **Zero-instrumentation** | Works out of the box — just add `MeshAdminConfig()` and connect |
| 3 | **Instant diagnostics** | One-key (`d`) full health check across the entire mesh |
| 4 | **Live debugging** | Py-spy stack traces (`p`) for diagnosing hangs, deadlocks, GIL contention |
| 5 | **Terminal-native** | No browser, no port forwarding, no web UI setup — pure terminal |
| 6 | **CI/CD integration** | `--diagnose` mode for scripted health checks with JSON output |
| 7 | **Failed-actor tracking** | Failed actors always remain visible (TUI-4), with terminated snapshots |
| 8 | **Config inspection** | View process configuration with source provenance |
| 9 | **Flight recorder** | Per-actor activity log for debugging message ordering |
| 10 | **Uniform API** | Same `NodePayload` for everything — no special cases in the client |
| 11 | **Lazy fetching** | Only loads what you expand — efficient even for large meshes |
| 12 | **TLS/mTLS support** | Secure connections with auto-detected or explicit certificates |
| 13 | **i18n** | English and Simplified Chinese out of the box |
| 14 | **Algebraic correctness** | 21+ formal invariants for reliable, predictable behavior |

---
## 11. Where Is the Admin TUI Useful?

| Scenario | How the TUI Helps |
|----------|-------------------|
| **Developing actor-based training pipelines** | See your actors spawn, receive messages, and interact in real time |
| **Debugging distributed hangs** | Py-spy captures live Python stack traces, showing exactly where each thread is stuck |
| **Post-failure investigation** | Failed and terminated actors remain visible with their last known state |
| **Multi-host cluster monitoring** | Navigate across hosts, see proc counts, actor health at a glance |
| **Production health checks** | `--diagnose` gives a scripted JSON report for automated monitoring |
| **Onboarding & demos** | The tree view makes Monarch's actor model tangible and visual |
| **Config auditing** | Inspect process configuration to verify correct settings per proc |
| **CI/CD pipeline gates** | Non-interactive diagnostics as a pass/fail check before deployment |

---
## 12. Comparison with Alternatives

| Feature | Admin TUI | Web Dashboard | Manual curl/logs |
|---------|-----------|---------------|------------------|
| Real-time topology tree | Yes | Yes | No (flat JSON) |
| Terminal-native (no browser) | Yes | No | Yes |
| Py-spy stack traces | Yes | No | Manual |
| Config inspection | Yes | No | Manual |
| Diagnostics overlay | Yes | No | Manual |
| Non-interactive mode | Yes (`--diagnose`) | No | Yes |
| Flight recorder events | Yes | Partial | No |
| Setup required | `pip install torchmonarch` | Port forwarding, browser | None |
| Works over SSH | Yes (native terminal) | Requires tunneling | Yes |

---
## 13. Source Code Reference

### TUI Crate
```
hyperactor_mesh_admin_tui/
├── Cargo.toml              # Crate manifest (ratatui, crossterm, reqwest, clap, ...)
├── bin/
│   └── admin_tui.rs        # Binary entry point — CLI args, TuiConfig, run()
└── src/
    ├── lib.rs              # Library root: terminal setup, TuiConfig, run(), 21 invariants
    ├── app.rs              # App struct, run_app() event loop, on_key() dispatcher
    ├── model.rs            # TreeNode, Cursor, NodeType, FlatRow, VisibleRows
    ├── tree.rs             # flatten_tree, fold_tree, collapse_all, tree algebra
    ├── fetch.rs            # FetchState<T>, join-semilattice, build_tree_node
    ├── filter.rs           # is_stopped_node, is_failed_node, is_system_node
    ├── format.rs           # Label derivation, time formatting
    ├── actions.rs          # KeyResult enum
    ├── job.rs              # ActiveJob (Diagnostics/PySpy/Config), overlay lifecycle
    ├── overlay.rs          # Scrollable overlay pane
    ├── client.rs           # TLS-aware reqwest client construction
    ├── diagnostics.rs      # Full diagnostic suite: DiagResult, DiagPhase, DiagOutcome
    ├── theme.rs            # Nord/DoomNordLight themes, En/Zh localization
    ├── render.rs           # Top-level layout (header + body + footer)
    ├── render/
    │   ├── tree_pane.rs    # Left pane: topology tree rendering
    │   ├── detail_pane.rs  # Right pane: contextual details + overlays
    │   └── status_bar.rs   # Header and footer bars
    └── tests/
        └── mod.rs          # ~2300 lines of integration tests
```

### Server Side (Admin HTTP API)
```
hyperactor_mesh/
├── src/
│   ├── mesh_admin.rs           # MeshAdminAgent: HTTP server, resolver, bridge
│   ├── mesh_admin_client.rs    # Shared TLS configuration
│   └── mesh_admin_skill.md     # Machine-readable API contract (SKILL.md)
└── test/
    └── mesh_admin_integration/ # Integration tests
```

### Documentation & Assets
```
docs/source/
├── admin-tui.md                                           # User-facing docs
├── books/hyperactor-mesh-book/src/meshes/
│   └── mesh-introspection-and-admin.md                    # Deep architectural overview
└── _static/
    ├── tui-tree.png                                       # Screenshot: topology tree
    ├── tui-diagnostics.png                                # Screenshot: diagnostics
    └── tui-pyspy.png                                      # Screenshot: py-spy
```

### Examples
```
python/examples/dining_philosophers.py       # Python dining philosophers (with admin TUI)
hyperactor_mesh/examples/dining_philosophers.rs  # Rust dining philosophers
```

---
## 14. End-to-End Call Flow: What Happens When You Expand a Node?

```
User presses Tab on a host node
  |
  v
on_key(Tab) -> KeyResult::ExpandNode
  |
  v
expand_node() -> for each child reference:
  |
  v
GET /v1/{child_reference}
  |
  v
MeshAdminAgent::resolve_reference(child_ref)
  |-- "host:..." -> HostAgent introspect query
  |-- proc_id    -> HostAgent QueryChild, else ProcAgent
  |-- actor_id   -> direct actor IntrospectMessage
  v
IntrospectResult { attrs, children }
  |
  v
derive_properties() -> NodeProperties
  |
  v
NodePayload { identity, properties, children, parent, as_of }
  |
  v
fetch_with_join() -> merge into FetchState cache (join-semilattice)
  |
  v
build_tree_node() -> recursive TreeNode { children: Vec<TreeNode> }
  |
  v
flatten_tree() -> Vec<FlatRow> (visible rows)
  |
  v
render via ratatui -> tree_pane + detail_pane + status_bar
```

---
## 15. Tips & Best Practices

1. **Start with `s` to hide system actors** — Focus on your own actors first, toggle system
   actors when you need to debug infrastructure.

2. **Use `d` early and often** — Diagnostics is your first-line health check. It probes every
   node and tells you exactly what's broken and where.

3. **Py-spy for hangs** — When a training job appears stuck, press `p` on the proc. The stack
   trace shows exactly where each thread is blocked (CUDA call, GIL, C extension, etc.).

4. **Failed actors are always visible** — You don't need to toggle filters to find crashes.
   Failed actors are color-coded and always shown (TUI-4 invariant).

5. **Use `--diagnose` in CI** — Run `monarch-tui --addr $ADDR --diagnose` as a post-deploy
   health gate. JSON output + exit code 0/1 integrates with any CI system.

6. **Config dump for debugging misconfigs** — Press `C` to see exactly what config values a proc
   is using and where each value came from (default, env var, file, etc.).

7. **Refresh rate tuning** — For large meshes, increase `--refresh-ms` to reduce API load.
   For demos, decrease it for snappier updates.

---

*Generated from the Monarch repository at `monarch_dev/`. See the source files listed in Section 13 for the most up-to-date implementation details.*